# SemiMind AI – Phase 1: SECOM Initial Analysis

> **Phase 1 | Notebook 3 of 3**  
> Dataset: SECOM Semiconductor Manufacturing Dataset  
> Source: https://archive.ics.uci.edu/dataset/179/secom

---

### Dataset Overview

SECOM represents a real-world semiconductor manufacturing monitoring system.  
It contains high-dimensional sensor measurements collected from a wafer fabrication process.

**File structure discovered during Phase 1 inspection:**

| File | Description | Delimiter | Header |
|------|-------------|-----------|--------|
| `secom.data` | Feature matrix — 1,567 rows × 590 features | Space | None |
| `secom_labels.data` | Labels + timestamp — 1,567 rows × 2 columns | Space | None |

**Label encoding:** `-1` = Pass (yield OK), `1` = Fail (yield excursion)


In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (10, 5)
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False
import seaborn as sns
sns.set_theme(style='whitegrid', palette='muted')

from src.utils.config import load_config
from src.utils.logger import setup_logging
from src.ingestion.dataset_loader import load_secom_from_config

setup_logging(level='WARNING', log_to_file=False)
cfg = load_config()

features_df, labels_df = load_secom_from_config(cfg)
print(f'SECOM features loaded → shape: {features_df.shape}')
print(f'SECOM labels loaded   → shape: {labels_df.shape}')

## Feature and Target Structure

In [ ]:
print(f'Feature matrix shape : {features_df.shape}')
print(f'Labels shape         : {labels_df.shape}')
print(f'First feature columns: {features_df.columns[:5].tolist()}')
print(f'Last feature columns : {features_df.columns[-5:].tolist()}')
print(f'Label columns        : {labels_df.columns.tolist()}')
print(f'\nFirst 5 rows of features (first 8 columns):')
display(features_df.iloc[:5, :8])

## Data Types

In [ ]:
dtype_counts = features_df.dtypes.value_counts()
print('Data type distribution across 590 feature columns:')
print(dtype_counts.to_string())

print(f'\nLabel dtype: {labels_df["label"].dtype}')
print(f'Timestamp dtype: {labels_df["timestamp"].dtype}')
print(f'\nSample timestamps:')
print(labels_df['timestamp'].head().to_string())

## Missing Value Analysis

In [ ]:
missing_counts = features_df.isnull().sum()
missing_pct = (missing_counts / len(features_df) * 100).round(2)
total_missing = missing_counts.sum()
cols_with_missing = (missing_counts > 0).sum()
fully_missing = (missing_counts == len(features_df)).sum()

print(f'Total missing cells      : {total_missing:,}')
print(f'Total cells              : {features_df.size:,}')
print(f'Overall missing %        : {total_missing / features_df.size * 100:.2f}%')
print(f'Columns with any missing : {cols_with_missing} / {features_df.shape[1]}')
print(f'Columns 100% missing     : {fully_missing}')

In [ ]:
# Distribution of missing percentages per column
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(missing_pct[missing_pct > 0], bins=30, color='tomato', edgecolor='white', alpha=0.85)
ax.set_title(f'SECOM – Missing Value % per Column ({cols_with_missing} columns affected)')
ax.set_xlabel('Missing %')
ax.set_ylabel('Number of columns')
plt.tight_layout()
plt.savefig(str(project_root / 'datasets' / 'reports' / 'secom' / 'secom_missing_distribution.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Top 20 most missing columns
top_missing = missing_pct[missing_pct > 0].nlargest(20)
print('Top 20 columns by missing %:')
display(pd.DataFrame({'missing_pct': top_missing}))

## Class Distribution

In [ ]:
label_vc = labels_df['label'].value_counts()
label_map = {-1: 'Pass (-1)', 1: 'Fail (1)'}

print('SECOM label distribution:')
for val, count in label_vc.items():
    pct = count / len(labels_df) * 100
    print(f'  {label_map[val]:12s} : {count:5d}  ({pct:.2f}%)')

imbalance_ratio = label_vc.max() / label_vc.min()
print(f'\nImbalance ratio: {imbalance_ratio:.1f}:1 (Pass:Fail)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
axes[0].bar(['Pass (-1)', 'Fail (1)'], label_vc.values, color=['steelblue', 'tomato'])
axes[0].set_title('SECOM – Class Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(label_vc.values):
    axes[0].text(i, v + 10, str(v), ha='center')

# Pie chart
axes[1].pie(label_vc.values, labels=['Pass', 'Fail'],
            colors=['steelblue', 'tomato'], autopct='%1.1f%%', startangle=90)
axes[1].set_title('SECOM – Class Proportions')

plt.tight_layout()
plt.savefig(str(project_root / 'datasets' / 'reports' / 'secom' / 'secom_class_distribution.png'), dpi=120, bbox_inches='tight')
plt.show()

## Constant Feature Identification

In [ ]:
constant_features = [
    col for col in features_df.columns
    if features_df[col].nunique(dropna=True) <= 1
]
print(f'Constant or all-NaN features: {len(constant_features)}')
if constant_features:
    print('Constant feature columns:')
    for c in constant_features:
        nunique = features_df[c].nunique(dropna=True)
        missing = features_df[c].isnull().sum()
        print(f'  {c:15s}  unique (non-null): {nunique}  missing: {missing}')

## Basic Visualisations – Feature Value Distributions (Sample)

In [ ]:
# Sample 8 features with least missing values for initial visualisation
least_missing = missing_counts.nsmallest(8).index.tolist()

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, col in enumerate(least_missing):
    data = features_df[col].dropna()
    axes[i].hist(data, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xlabel('Value', fontsize=8)
    axes[i].set_ylabel('Frequency', fontsize=8)

plt.suptitle('SECOM – Sample Feature Distributions (8 features with fewest missing)', fontsize=12)
plt.tight_layout()
plt.savefig(str(project_root / 'datasets' / 'reports' / 'secom' / 'secom_feature_distributions.png'), dpi=120, bbox_inches='tight')
plt.show()

## Numerical Summary (Sample Features)

In [ ]:
# Show describe for first 10 features
display(features_df.iloc[:, :10].describe().round(3))

## Initial Observations

**Findings from SECOM initial inspection (Phase 1):**

1. **Severe class imbalance** — 93.4% Pass vs 6.6% Fail (ratio ≈ 14:1). Standard accuracy metrics will be misleading.
2. **High dimensionality** — 590 features for 1,567 samples. Classic *p >> n* scenario. Feature selection is critical.
3. **Pervasive missing values** — 41,951 total missing cells across 538 of 590 columns (avg ~7.2% per column).
4. **Constant / zero-variance features** — Some features have no non-null distinct values. These will be dropped in Phase 2 preprocessing.
5. **All features are float64** — Consistent numeric types; no categorical sensor readings.
6. **Timestamps are available** — Enables temporal ordering and time-series approaches in later phases.
7. **No column names in source file** — Auto-generated `feature_001 … feature_590` naming has been applied.

**Phase 2 priorities for this dataset:**
- Impute or drop missing values with a documented strategy.
- Remove constant features.
- Apply variance-threshold and correlation-based feature screening.
- Address severe class imbalance (SMOTE, class weighting, or threshold-moving).
- Parse timestamps for chronological train/test splitting.
